# Artigo 0007 - Agentes e Tool Calling

Roteamento de ferramentas, retrieval e resposta final usando modelos gratuitos do Hugging Face.

## Modelos usados
- Embeddings para roteamento e retrieval: `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`
- Geração da resposta final: `Qwen/Qwen2.5-0.5B-Instruct`

Aqui mostramos um agente local sem API paga nem chave externa.

In [1]:
from pathlib import Path
import sys
import json

# Faz o notebook funcionar tanto na pasta do artigo quanto via raiz do repositório.
ARTICLE_ROOT = Path.cwd()
if not (ARTICLE_ROOT / "src").exists():
    ARTICLE_ROOT = ARTICLE_ROOT.parent
sys.path.insert(0, str(ARTICLE_ROOT))

import pandas as pd
import matplotlib.pyplot as plt
from src.agent_runner import (
    DEFAULT_EMBEDDING_MODEL,
    DEFAULT_GENERATION_MODEL,
    run_agent,
)

In [ ]:
# Executa o agente em um conjunto pequeno de cenários para inspeção detalhada.
# O runner também grava arquivos auxiliares com auditoria, resumo de decisão
# e recorte específico do retrieval.
resultados, resumo = run_agent(limit=4)
print("Modelo de geracao:", DEFAULT_GENERATION_MODEL)
print("Modelo de embedding:", DEFAULT_EMBEDDING_MODEL)
resumo


## O que mudou no desenho do agente

Este notebook não mostra apenas a resposta final. Ele expõe a cadeia de decisão:

1. guardrail
2. regra de negócio
3. planner em JSON
4. fallback por embeddings
5. execução da tool
6. resposta final

In [ ]:
resultados[["entrada", "expected_action", "planned_tool", "acao", "used_fallback", "documento", "resposta"]]


In [ ]:
resultados[["entrada", "planned_tool", "plan_reason", "router_score", "knowledge_score", "acertou_roteamento"]]

In [ ]:
contagem = resultados["acao"].value_counts().rename_axis("acao").reset_index(name="total")
contagem

In [ ]:
plt.figure(figsize=(8, 4))
plt.bar(contagem["acao"], contagem["total"])
plt.title("Distribuicao de acoes do agente local")
plt.ylabel("Total")
plt.xticks(rotation=20)
plt.show()

resultados[["entrada", "audit_steps"]].head(2)

## Tool registry e base de conhecimento

Antes de olhar a decisão final do agente, vale inspecionar os dois ativos que condicionam o comportamento dele:

- o catálogo de tools disponíveis;
- a base documental usada pelo retrieval.

In [ ]:
tool_registry = pd.read_json(ARTICLE_ROOT / "data" / "tool_registry.json")
knowledge_base = pd.read_json(ARTICLE_ROOT / "data" / "knowledge_base.json")

tool_registry, knowledge_base[["id", "titulo"]]

## Artefatos auxiliares do agente

Além da tabela principal, o runner salva outros arquivos que ajudam a inspecionar a maturidade operacional do agente:

- `decision_summary.csv`
- `retrieval_summary.csv`
- `agent_audit_log.jsonl`

In [ ]:
decision_summary = pd.read_csv(ARTICLE_ROOT / "data" / "decision_summary.csv")
retrieval_summary = pd.read_csv(ARTICLE_ROOT / "data" / "retrieval_summary.csv")

decision_summary, retrieval_summary

In [ ]:
# Audit trail de um cenário. Esse bloco mostra o que um time de engenharia
# realmente precisa revisar quando o agente toma uma decisão inesperada.
audit_steps = json.loads(resultados.loc[0, "audit_steps"])
pd.DataFrame(audit_steps)